In [ ]:
import transformers
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Running on: {device}")

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification


model_id = "distilbert/distilbert-base-uncased-finetuned-sst-2-english"

# Underthe hood uses tensors 
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForSequenceClassification.from_pretrained(model_id).to(device)



In [ ]:
text = "Oh great, another broken screen. Just what I always wanted"

inputs = tokenizer(text, return_tensors="pt").to(device)

with torch.inference_mode():
    outputs = model(**inputs)
# logits
print(outputs)

# predicted class
predicted_class = outputs.logits.argmax(dim=-1).item()
print(model.config.id2label[predicted_class])


## Text Generating and Streaming

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM,  TextStreamer
import torch

# check gpu compatibility
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# device = torch.device("cpu")

# load tokenizer and model
model_id = "Qwen/Qwen3-0.6B"
tokenizer = AutoTokenizer.from_pretrained(model_id, return_tensors="pt")
model = AutoModelForCausalLM.from_pretrained(model_id).to(device)
streamer = TextStreamer(tokenizer=tokenizer, skip_prompt=True)

In [ ]:
# Text Generation
text = """
write a story on rich people
"""
inputs = tokenizer(text, return_tensors="pt").to(device)

with torch.inference_mode():
    # Probablistic sampling instead of greedy_sampling
    outputs = model.generate(
                                **inputs,
                                temperature=0.7,
                                do_sample=True,
                                max_new_tokens=200,
                                top_p=0.9,
                                repetition_penalty=1.4,
                            )
res = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(res) 

In [ ]:
# Streaming
prompt = "what is 5 plu 8."
inputs = tokenizer(
    prompt,
    return_tensors = "pt"
)

with torch.inference_mode():
    model.generate(
        **inputs,
        top_p=0.9,
        repetition_penalty=1.3,
        temperature=.7,
        max_new_tokens=500,
        do_sample=True,
        streamer=streamer,
        eos_token_id=tokenizer.eos_token_id
    )